# DipRadar ML v1.0 — Backtest

Valida se o modelo tem edge **real** em condições simuladas de trading.  
Não é um backtest de execução — é um backtest de **selecção**: o modelo escolhe quais alertas activar.

**Inputs:**  
- `dataset_v3.parquet` — features + `max_return_60d` + `max_drawdown_60d`  
- `model_up_v1.joblib` + `model_down_v1.joblib` — modelos treinados  

**Outputs:**  
- PnL por trade, Sharpe, Max Drawdown, Win Rate  
- Análise por regime (VIX, bull/bear)  
- Curva de equity acumulada  


## 0. Instalações

In [ ]:
!pip install xgboost scikit-learn pandas numpy scipy joblib matplotlib seaborn --quiet

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import joblib, json, warnings
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr, mstats
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_style('darkgrid')

# ── Paths ──
DATA_PATH  = Path('/content/DipRadar/experiments/ml_v2/dataset_v3.parquet')
MODEL_DIR  = Path('/content/DipRadar/models')
OUT_DIR    = Path('/content/DipRadar/experiments/ml_v2/backtest_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Targets ──
TARGET_UP   = 'max_return_60d'
TARGET_DOWN = 'max_drawdown_60d'

# ── Backtest params ──
SCORE_THRESHOLDS = [0.5, 1.0, 1.5, 2.0, 2.5]  # testar varios thresholds
TP = 0.30   # take profit +30%
SL = -0.15  # stop loss   -15%
HOLDING_DAYS = 60

# ── Feature set (deve coincidir com v1_Production) ──
FEATURES = [
    'macro_score', 'vix', 'vix_regime', 'sp500_trend',
    'drop_pct_today', 'drawdown_52w', 'distance_from_ma50',
    'return_5d', 'return_10d', 'return_20d', 'return_1m', 'return_3m_pre',
    'zscore_20d', 'atr_ratio', 'volatility_20d', 'beta_60d',
    'volume_spike', 'spy_drawdown_5d', 'sector_drawdown_5d', 'sector_relative',
    'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside',
    'quality_score', 'pe_attractive', 'drop_x_drawdown', 'vol_x_drop',
    'rsi_14', 'rsi_oversold_strength',
]

print('Config OK')
print(f'TP={TP:.0%}  SL={SL:.0%}  holding={HOLDING_DAYS}d')

## 2. Funções Auxiliares

In [ ]:
def log1p_signed(x):  return np.sign(x) * np.log1p(np.abs(x))
def inv_log1p_signed(x): return np.sign(x) * np.expm1(np.abs(x))
def winsorize_col(arr, limits=(0.01, 0.01)): return mstats.winsorize(arr, limits=limits).data
def transform_targets(u, d): return log1p_signed(u), log1p_signed(d)
def inverse_transform(u, d): return inv_log1p_signed(u), inv_log1p_signed(d)

def compute_score(pred_up, pred_down, eps=0.01):
    return pred_up / (np.abs(pred_down) + eps)

def make_model_up():
    return XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
        reg_lambda=1.0, random_state=42, n_jobs=-1, verbosity=0)

def make_model_down():
    return XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.2,
        reg_lambda=2.0, random_state=42, n_jobs=-1, verbosity=0)

def simulate_trade(row, tp=TP, sl=SL):
    """
    Simula PnL de um trade dado max_return e max_drawdown no holding period.
    Assume que TP e SL sao atingidos proporcionalmente ao maximo observado.
    """
    r   = row[TARGET_UP]    # maximo retorno observado em 60d
    dd  = row[TARGET_DOWN]  # maximo drawdown observado em 60d
    # Hit TP?
    if r >= tp:   return tp
    # Hit SL?
    if dd <= sl:  return sl
    # Sem hit -> retorno final = max_return (proxy)
    return r

print('Funções definidas.')

## 3. Carregar Dataset

In [ ]:
df = pd.read_parquet(DATA_PATH)
df['alert_date'] = pd.to_datetime(df['alert_date'])
df = df.sort_values('alert_date').reset_index(drop=True)
FEATURES = [f for f in FEATURES if f in df.columns]

print(f'Dataset: {len(df):,} amostras | {df["alert_date"].min().date()} -> {df["alert_date"].max().date()}')
print(f'Features: {len(FEATURES)}')

# Baseline: retorno médio de todos os alertas (sem filtro)
baseline_mean = df[TARGET_UP].mean()
baseline_med  = df[TARGET_UP].median()
print(f'Baseline mean={baseline_mean:.4f}  median={baseline_med:.4f}')

## 4. Walk-Forward: Gerar Previsões Out-of-Sample

Re-treina em cada fold e guarda as previsões do conjunto de teste.  
Nunca usa dados futuros para treinar — sem lookahead bias.

In [ ]:
N_FOLDS   = 5
TEST_SIZE = int(len(df) * 0.12)
tscv = TimeSeriesSplit(n_splits=N_FOLDS, test_size=TEST_SIZE)

all_preds = []

print('A gerar previsoes OOS...')
for k, (tr_idx, te_idx) in enumerate(tscv.split(df)):
    df_tr = df.iloc[tr_idx]
    df_te = df.iloc[te_idx].copy()
    X_tr  = df_tr[FEATURES].fillna(0).values.astype(np.float32)
    X_te  = df_te[FEATURES].fillna(0).values.astype(np.float32)
    y_up_tr   = winsorize_col(df_tr[TARGET_UP].fillna(0).values)
    y_down_tr = winsorize_col(df_tr[TARGET_DOWN].fillna(0).values)
    y_up_t, y_down_t = transform_targets(y_up_tr, y_down_tr)
    m_up   = make_model_up();   m_up.fit(X_tr, y_up_t)
    m_down = make_model_down(); m_down.fit(X_tr, y_down_t)
    pred_up, pred_down = inverse_transform(m_up.predict(X_te), m_down.predict(X_te))
    df_te['pred_up']   = pred_up
    df_te['pred_down'] = pred_down
    df_te['score']     = compute_score(pred_up, pred_down)
    df_te['fold']      = k + 1
    all_preds.append(df_te)
    print(f'  Fold {k+1} OK | n_test={len(te_idx)}')

df_bt = pd.concat(all_preds).reset_index(drop=True)
df_bt['pnl_tpsl'] = df_bt.apply(simulate_trade, axis=1)
df_bt['date_year'] = df_bt['alert_date'].dt.year

print(f'\nTotal alertas OOS: {len(df_bt):,}')
print(f'PnL médio (sem filtro): {df_bt["pnl_tpsl"].mean():.4f}')

## 5. Análise por Score Threshold

In [ ]:
def sharpe(pnl_series, periods_per_year=252/60):
    if pnl_series.std() == 0: return 0.0
    return float(pnl_series.mean() / pnl_series.std() * np.sqrt(periods_per_year))

def max_drawdown(pnl_series):
    equity = (1 + pnl_series).cumprod()
    peak   = equity.cummax()
    dd     = (equity - peak) / peak
    return float(dd.min())

threshold_results = []
baseline_pnl = df_bt['pnl_tpsl']

for thr in SCORE_THRESHOLDS:
    sel = df_bt[df_bt['score'] >= thr]['pnl_tpsl']
    if len(sel) < 10:
        continue
    threshold_results.append({
        'threshold':   thr,
        'n_trades':    len(sel),
        'pct_alertas': len(sel) / len(df_bt),
        'mean_pnl':    sel.mean(),
        'median_pnl':  sel.median(),
        'win_rate':    (sel > 0).mean(),
        'sharpe':      sharpe(sel),
        'max_dd':      max_drawdown(sel),
        'vs_baseline': sel.mean() - baseline_pnl.mean(),
    })

df_thr = pd.DataFrame(threshold_results)
print('=== Resultados por Threshold ===\n')
print(df_thr.round(4).to_string(index=False))

## 6. Regime Split (VIX e Bull/Bear)

In [ ]:
# Usa threshold=1.5 (BUY default)
THR = 1.5
df_sel = df_bt[df_bt['score'] >= THR].copy()

# ── Regime VIX ──
df_sel['vix_regime_label'] = pd.cut(
    df_sel['vix'], bins=[0, 15, 25, 40, 999],
    labels=['Low VIX (<15)', 'Normal (15-25)', 'High (25-40)', 'Extreme (>40)']
)

vix_regimes = df_sel.groupby('vix_regime_label', observed=True)['pnl_tpsl'].agg(
    n='count', mean='mean', win_rate=lambda x: (x>0).mean(), sharpe=sharpe
).round(4)
print('=== Performance por Regime VIX ===')
print(vix_regimes.to_string())

# ── Regime Bull/Bear (proxy: spy_drawdown_5d) ──
if 'spy_drawdown_5d' in df_sel.columns:
    df_sel['market_regime'] = np.where(df_sel['spy_drawdown_5d'] < -0.03, 'Bear/Stress', 'Bull/Normal')
    bull_bear = df_sel.groupby('market_regime')['pnl_tpsl'].agg(
        n='count', mean='mean', win_rate=lambda x: (x>0).mean(), sharpe=sharpe
    ).round(4)
    print('\n=== Performance Bull vs Bear ===')
    print(bull_bear.to_string())

## 7. Curva de Equity Acumulada

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('DipRadar v1.0 — Backtest (OOS Walk-Forward)', fontsize=14, fontweight='bold')

# ── 1. Equity curves por threshold ──
ax = axes[0, 0]
df_bt_sorted = df_bt.sort_values('alert_date')
for thr in [0.0, 1.0, 1.5, 2.0]:
    sel = df_bt_sorted[df_bt_sorted['score'] >= thr]['pnl_tpsl'] if thr > 0 else df_bt_sorted['pnl_tpsl']
    if len(sel) < 10: continue
    equity = (1 + sel.reset_index(drop=True)).cumprod()
    label  = f'score>={thr}  (n={len(sel)})' if thr > 0 else f'Baseline (n={len(sel)})'
    ax.plot(equity.values, label=label)
ax.set_title('Equity Curve (por threshold)')
ax.set_xlabel('Trade #')
ax.set_ylabel('Equity (1=start)')
ax.legend(fontsize=8)

# ── 2. Distribuição PnL score>=1.5 ──
ax = axes[0, 1]
sel_15 = df_bt[df_bt['score'] >= 1.5]['pnl_tpsl']
ax.hist(sel_15, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(sel_15.mean(), color='red', linestyle='--', label=f'mean={sel_15.mean():.3f}')
ax.axvline(0, color='black', linestyle='-', linewidth=0.8)
ax.set_title('Distribuição PnL (score >= 1.5)')
ax.set_xlabel('PnL por trade')
ax.legend()

# ── 3. Mean PnL por threshold ──
ax = axes[1, 0]
ax.bar(df_thr['threshold'].astype(str), df_thr['mean_pnl'], color='steelblue')
ax.axhline(baseline_pnl.mean(), color='red', linestyle='--', label=f'baseline={baseline_pnl.mean():.3f}')
ax.set_title('Mean PnL por Score Threshold')
ax.set_xlabel('Threshold')
ax.set_ylabel('Mean PnL')
ax.legend()

# ── 4. Sharpe por threshold ──
ax = axes[1, 1]
ax.bar(df_thr['threshold'].astype(str), df_thr['sharpe'], color='seagreen')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Sharpe Ratio por Score Threshold')
ax.set_xlabel('Threshold')
ax.set_ylabel('Sharpe')

plt.tight_layout()
plt.savefig(OUT_DIR / 'backtest_summary.png', dpi=130)
plt.show()
print('Gráfico guardado.')

## 8. Análise por Ano

In [ ]:
df_sel_15 = df_bt[df_bt['score'] >= 1.5].copy()

yearly = df_sel_15.groupby('date_year')['pnl_tpsl'].agg(
    n='count',
    mean='mean',
    win_rate=lambda x: (x > 0).mean(),
    sharpe=sharpe,
    max_dd=max_drawdown
).round(4)

print('=== Performance por Ano (score >= 1.5) ===')
print(yearly.to_string())

# Baseline por ano
yearly_base = df_bt.groupby('date_year')['pnl_tpsl'].mean().round(4)
print('\n=== Baseline por Ano (sem filtro) ===')
print(yearly_base.to_string())

## 9. Diagnóstico Final & Guardar Resultados

In [ ]:
# Guardar previsoes e metricas
df_bt.to_parquet(OUT_DIR / 'oos_predictions.parquet', index=False)
df_thr.to_csv(OUT_DIR / 'threshold_metrics.csv', index=False)
yearly.to_csv(OUT_DIR / 'yearly_metrics.csv')

# Threshold escolhido
best_row = df_thr.loc[df_thr['sharpe'].idxmax()]
print('=== DIAGNÓSTICO FINAL ===')
print(f'Baseline mean PnL    : {baseline_pnl.mean():.4f}')
print(f'Melhor threshold     : score >= {best_row["threshold"]}')
print(f'  n_trades           : {int(best_row["n_trades"])}  ({best_row["pct_alertas"]:.1%} dos alertas)')
print(f'  mean_pnl           : {best_row["mean_pnl"]:.4f}')
print(f'  win_rate           : {best_row["win_rate"]:.4f}')
print(f'  sharpe             : {best_row["sharpe"]:.4f}')
print(f'  max_drawdown       : {best_row["max_dd"]:.4f}')
print(f'  vs_baseline        : +{best_row["vs_baseline"]:.4f}')

print('\nFicheiros guardados em:', OUT_DIR)

## 10. Notas de Interpretação

### O que valida o modelo
- `mean_pnl(score>=1.5) >> baseline` → o filtro acrescenta valor
- `sharpe > 0.5` → retornos consistentes (não só spikes)
- `win_rate > 55%` → mais de metade dos trades são positivos
- Performance estável em bull **e** bear → não é bias de mercado

### Red flags
- `sharpe` colapsa num regime (ex: só funciona em high-VIX) → cuidado
- `max_drawdown < -50%` → risco de ruína em sequência de perdas
- Performance 2020–2021 muito melhor que 2022–2026 → overfit ao COVID

### Próximo passo
Se Sharpe > 0.5 e estável por regime:  
→ integrar `score_alert()` no bot de alertas com `threshold=best_threshold`
